# BC Detection Benchmark Analysis 

In [1]:
import pandas as pd
from pandas.api.typing import NAType
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from typing import Any, Literal

## Accuracy Statistics

In [2]:
def to_bool(value: Any) -> bool | NAType:
    """
    Convert a value to a boolean or ``pd.NA``.
    :param v: A value that may be a boolean, string, or missing value.
    :returns: The boolean value, or ``pd.NA`` if the input is missing or not
        recognized as a boolean-like value.
    """
    if pd.isna(value):
        return pd.NA
    if isinstance(value, bool):
        return value
    str_value = str(value).strip().lower()
    if str_value == "true":
        return True
    if str_value == "false":
        return False
    return pd.NA

In [ ]:
base = Path("data")
files = [base / f"results-by-case-{i}.csv" for i in range(1, 6)]

runs = []
for run_idx, f in enumerate(files, start=1):
    df_run = pd.read_csv(f, sep=";")
    df_run["run"] = run_idx
    runs.append(df_run)

all_runs = pd.concat(runs, ignore_index=True)
all_runs

,case,bin,src,message,gpt-5.4_bin,gpt-5.4_src,correct_gpt-5.4_bin,correct_gpt-5.4_src,gpt-5.4_input_tokens,gpt-5.4_output_tokens,...,gpt-5.4_message,claude-opus-4.6_bin,claude-opus-4.6_src,correct_claude-opus-4.6_bin,correct_claude-opus-4.6_src,claude-opus-4.6_input_tokens,claude-opus-4.6_output_tokens,claude-opus-4.6_request_time,claude-opus-4.6_message,run
0,membersClazzMethodAdd,False,False,NaN,False,False,True,True,603,50,...,raw_response=Source incompatible: NO Binary i...,False,False,True,True,762,40,4248.563292,raw_response=``` Source incompatible: NO Binar...,1
1,dataTypeIfazeMethodParamSpecialization,True,True,Compiler: compiler.err.does.not.override.abstr...,True,True,True,True,637,277,...,raw_response=Source incompatible: YES Binary ...,True,True,True,True,803,126,6163.387417,raw_response=``` Source incompatible: YES Bina...,1
2,modifierClazzNonStrictfpToStrictfp,False,False,NaN,False,False,True,True,621,278,...,raw_response=Source incompatible: NO Binary i...,False,False,True,True,800,43,6797.455500,raw_response=``` Source incompatible: NO Binar...,1
3,genericsWildcardsClazzConstructorParamAdd,False,False,NaN,False,False,True,True,670,218,...,raw_response=Source incompatible: NO Binary i...,False,False,True,True,864,153,8688.509792,raw_response=Source incompatible: NO Binary i...,1
4,membersClazzMethodDelete,True,True,Compiler: compiler.err.cant.resolve.location.a...,True,True,True,True,603,61,...,raw_response=Source incompatible: YES Binary ...,True,True,True,True,762,35,2169.911583,raw_response=Source incompatible: YES Binary i...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1545,genericsClazzMethodTypeDeleteN,False,True,Compiler: compiler.err.name.clash.same.erasure...,False,True,True,True,636,265,...,raw_response=Source incompatible: YES Binary ...,False,False,True,False,792,355,10072.256625,raw_response=``` Source incompatible: NO Binar...,5
1546,genericsIfazeMethodTypeDeleteN,False,True,Compiler: compiler.err.does.not.override.abstr...,False,True,True,True,634,806,...,raw_response=Source incompatible: YES Binary ...,False,False,True,False,786,406,16524.933542,raw_response=Source incompatible: NO Binary i...,5
1547,inheritanceIfazeContractSuperinterfaceSet,True,True,"Compiler: compiler.err.cant.resolve.location, ...",True,True,True,True,722,1119,...,raw_response=Source incompatible: YES Binary ...,True,True,True,True,949,452,13149.171042,raw_response=Source incompatible: YES Binary ...,5
1548,inheritanceIfazeDefaultMethodOverrideAdd,False,False,NaN,False,False,True,True,758,2641,...,raw_response=Source incompatible: NO Binary i...,False,False,True,True,981,47,4416.338584,raw_response=Source incompatible: NO Binary in...,5


In [4]:
missing_counts = {
    col: int(all_runs[col].isna().sum())
    for col in all_runs.columns
}

missing_df = pd.DataFrame(
    {
        "column": list(missing_counts.keys()),
        "missing_values": list(missing_counts.values()),
    }
).sort_values("missing_values", ascending=False)

print("Missing values across all columns in all_runs:")
display(missing_df)

print(f"Total rows in all_runs: {len(all_runs)}")

Missing values across all columns in all_runs:


,column,missing_values
3,message,595
0,case,0
11,gpt-5.4_message,0
19,claude-opus-4.6_message,0
18,claude-opus-4.6_request_time,0
17,claude-opus-4.6_output_tokens,0
16,claude-opus-4.6_input_tokens,0
15,correct_claude-opus-4.6_src,0
14,correct_claude-opus-4.6_bin,0
13,claude-opus-4.6_src,0


Total rows in all_runs: 1550


In [5]:
def majority_vote(series: pd.Series) -> bool | NAType | Literal["TIE"]:
    """
    Return the majority vote value in a series of runs for a benchmark case.
    :param series: A pandas Series of runs for a benchmark case.
    :returns: The value with the highest frequency, ``pd.NA`` if the series
        contains only null values, or ``"TIE"`` if multiple values are tied
        for the highest frequency.
    """
    series = series.dropna()
    if series.empty:
        return pd.NA
    
    counts = series.value_counts()
    top = counts.iloc[0]
    winners = counts[counts == top].index.tolist()
    return winners[0] if len(winners) == 1 else "TIE"

In [6]:
vote_cols = [
    "gpt-5.4_src",
    "gpt-5.4_bin",
    "claude-opus-4.6_src",
    "claude-opus-4.6_bin",
]

correct_cols = [
    "correct_gpt-5.4_src",
    "correct_gpt-5.4_bin",
    "correct_claude-opus-4.6_src",
    "correct_claude-opus-4.6_bin",
]

majority_votes_df = (
    all_runs.groupby("case", as_index=False)[vote_cols + correct_cols]
    .agg(majority_vote)
    .sort_values("case")
    .reset_index(drop=True)
)
majority_votes_df

,case,gpt-5.4_src,gpt-5.4_bin,claude-opus-4.6_src,claude-opus-4.6_bin,correct_gpt-5.4_src,correct_gpt-5.4_bin,correct_claude-opus-4.6_src,correct_claude-opus-4.6_bin
0,accessModifierClazzAccessDecrease,True,True,True,True,True,True,True,True
1,accessModifierClazzAccessIncrease,False,False,False,False,True,True,True,True
2,accessModifierClazzConstructorAccessDecreaseNo...,True,True,True,True,False,False,False,False
3,accessModifierClazzConstructorAccessDecreasePr...,True,True,True,True,True,True,True,True
4,accessModifierClazzConstructorAccessDecreasePr...,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...
305,otherClazzDelete,True,True,True,True,True,True,True,True
306,otherClazzToIfaze,True,True,True,True,True,True,True,True
307,otherIfazeAdd,False,False,False,False,True,True,True,True
308,otherIfazeDelete,True,True,True,True,True,True,True,True


In [7]:
# Add breaking-change majority as OR(source, binary).
majority_votes_df["gpt-5.4_breaking"] = (
    majority_votes_df["gpt-5.4_src"].astype(str).str.lower().eq("true")
    | majority_votes_df["gpt-5.4_bin"].astype(str).str.lower().eq("true")
)
majority_votes_df["claude-opus-4.6_breaking"] = (
    majority_votes_df["claude-opus-4.6_src"].astype(str).str.lower().eq("true")
    | majority_votes_df["claude-opus-4.6_bin"].astype(str).str.lower().eq("true")
)

# Add whether majority decisions are mostly correct per case.
majority_votes_df["gpt-5.4_breaking_correct"] = (
    majority_votes_df["correct_gpt-5.4_src"].astype(str).str.lower().eq("true")
    & majority_votes_df["correct_gpt-5.4_bin"].astype(str).str.lower().eq("true")
)
majority_votes_df["claude-opus-4.6_breaking_correct"] = (
    majority_votes_df["correct_claude-opus-4.6_src"].astype(str).str.lower().eq("true")
    & majority_votes_df["correct_claude-opus-4.6_bin"].astype(str).str.lower().eq("true")
)

majority_votes_df


,case,gpt-5.4_src,gpt-5.4_bin,claude-opus-4.6_src,claude-opus-4.6_bin,correct_gpt-5.4_src,correct_gpt-5.4_bin,correct_claude-opus-4.6_src,correct_claude-opus-4.6_bin,gpt-5.4_breaking,claude-opus-4.6_breaking,gpt-5.4_breaking_correct,claude-opus-4.6_breaking_correct
0,accessModifierClazzAccessDecrease,True,True,True,True,True,True,True,True,True,True,True,True
1,accessModifierClazzAccessIncrease,False,False,False,False,True,True,True,True,False,False,True,True
2,accessModifierClazzConstructorAccessDecreaseNo...,True,True,True,True,False,False,False,False,True,True,False,False
3,accessModifierClazzConstructorAccessDecreasePr...,True,True,True,True,True,True,True,True,True,True,True,True
4,accessModifierClazzConstructorAccessDecreasePr...,True,True,True,True,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,otherClazzDelete,True,True,True,True,True,True,True,True,True,True,True,True
306,otherClazzToIfaze,True,True,True,True,True,True,True,True,True,True,True,True
307,otherIfazeAdd,False,False,False,False,True,True,True,True,False,False,True,True
308,otherIfazeDelete,True,True,True,True,True,True,True,True,True,True,True,True


In [8]:
def compute_accuracy(groundtruth: pd.Series, results: pd.Series) -> dict:
    """
    Compute precision, recall, and F1 of BC detections against ground-truth.
    :param groundtruth: A pandas Series of ground-truth boolean labels.
    :param results: A pandas Series of predicted boolean labels, aligned with ``groundtruth``.
    :returns: A dict with the summary of accuracy values.
    """
    mask = groundtruth.notna() & results.notna()
    yt = groundtruth[mask]
    yp = results[mask]

    tp = int(((yp == True) & (yt == True)).sum())
    fp = int(((yp == True) & (yt == False)).sum())
    fn = int(((yp == False) & (yt == True)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "evaluated_cases": int(mask.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [9]:
# Build ground truth per case from dataset labels.
groundtruth_df = (
    all_runs.groupby("case", as_index=False)[["src", "bin"]]
    .first()
    .rename(columns={"src": "truth_src", "bin": "truth_bin"})
)
groundtruth_df["truth_breaking"] = groundtruth_df["truth_src"] | groundtruth_df["truth_bin"]

groundtruth_df

,case,truth_src,truth_bin,truth_breaking
0,accessModifierClazzAccessDecrease,True,True,True
1,accessModifierClazzAccessIncrease,False,False,False
2,accessModifierClazzConstructorAccessDecreaseNo...,False,False,False
3,accessModifierClazzConstructorAccessDecreasePr...,True,True,True
4,accessModifierClazzConstructorAccessDecreasePr...,True,True,True
...,...,...,...,...
305,otherClazzDelete,True,True,True
306,otherClazzToIfaze,True,True,True
307,otherIfazeAdd,False,False,False
308,otherIfazeDelete,True,True,True


In [10]:
# Merge truth with majority-vote classifications.
eval_df = majority_votes_df.merge(groundtruth_df, on="case", how="inner")

# Normalize boolean-like values.
for c in [
    "gpt-5.4_src",
    "gpt-5.4_bin",
    "claude-opus-4.6_src",
    "claude-opus-4.6_bin",
]:
    eval_df[c] = eval_df[c].map(to_bool)

eval_df["gpt-5.4_breaking"] = eval_df["gpt-5.4_src"] | eval_df["gpt-5.4_bin"]
eval_df["claude-opus-4.6_breaking"] = eval_df["claude-opus-4.6_src"] | eval_df["claude-opus-4.6_bin"]

rows = []
for model in ["gpt-5.4", "claude-opus-4.6"]:
    for target, truth_col in [
        ("src", "truth_src"),
        ("bin", "truth_bin"),
        ("breaking", "truth_breaking"),
    ]:
        pred_col = f"{model}_{target}"
        metrics = compute_accuracy(eval_df[truth_col], eval_df[pred_col])
        rows.append({"model": model, "target": target, **metrics})

metrics_df = pd.DataFrame(rows).sort_values(["model", "target"]).reset_index(drop=True)

# Display metrics.
metrics_display_df = metrics_df.copy()
for c in ["precision", "recall", "f1"]:
    metrics_display_df[c] = metrics_display_df[c].map(lambda x: f"{x:.2g}")

metrics_display_df

,model,target,evaluated_cases,tp,fp,fn,precision,recall,f1
0,claude-opus-4.6,bin,310,104,7,1,0.94,0.99,0.96
1,claude-opus-4.6,breaking,310,179,8,12,0.96,0.94,0.95
2,claude-opus-4.6,src,310,159,17,12,0.9,0.93,0.92
3,gpt-5.4,bin,310,105,10,0,0.91,1,0.95
4,gpt-5.4,breaking,310,190,18,1,0.91,0.99,0.95
5,gpt-5.4,src,310,170,35,1,0.83,0.99,0.9


In [11]:
# Comparison table: highlight the best model for precision, recall, and f1 per target.
comparison_df = pd.DataFrame(
    {
        "target": ["src", "bin", "breaking"],
    }
)

for metric in ["precision", "recall", "f1"]:
    pivot = metrics_df.pivot(index="target", columns="model", values=metric)
    comparison_df[f"gpt-5.4_{metric}"] = comparison_df["target"].map(pivot["gpt-5.4"])
    comparison_df[f"claude-opus-4.6_{metric}"] = comparison_df["target"].map(pivot["claude-opus-4.6"])
    comparison_df[f"best_{metric}_model"] = comparison_df.apply(
        lambda r: "TIE"
        if pd.isna(r[f"gpt-5.4_{metric}"]) or pd.isna(r[f"claude-opus-4.6_{metric}"]) or r[f"gpt-5.4_{metric}"] == r[f"claude-opus-4.6_{metric}"]
        else ("gpt-5.4" if r[f"gpt-5.4_{metric}"] > r[f"claude-opus-4.6_{metric}"] else "claude-opus-4.6"),
        axis=1,
    )


def highlight_best(row: pd.Series) -> list[str]:
    """
    Return per-column CSS styles highlighting the best-performing model for a comparison row.

    :param row: A row of ``comparison_df``, containing the per-model metric columns
        (e.g. ``gpt-5.4_precision``, ``claude-opus-4.6_precision``) and the corresponding
        ``best_<metric>_model`` columns identifying which model won each metric (or ``"TIE"``).
    :returns: A list of CSS style strings, one per column of ``comparison_df`` in order.
        The cell of the winning model for each metric gets a highlight style; every other
        cell (including both cells on a ``"TIE"``) gets an empty string.
    """
    styles = [""] * len(comparison_df.columns)
    col_index = {col: i for i, col in enumerate(comparison_df.columns)}

    if row["best_precision_model"] == "gpt-5.4":
        styles[col_index["gpt-5.4_precision"]] = "background-color: #d9f2d9; font-weight: 700;"
    elif row["best_precision_model"] == "claude-opus-4.6":
        styles[col_index["claude-opus-4.6_precision"]] = "background-color: #d9f2d9; font-weight: 700;"

    if row["best_recall_model"] == "gpt-5.4":
        styles[col_index["gpt-5.4_recall"]] = "background-color: #d9f2d9; font-weight: 700;"
    elif row["best_recall_model"] == "claude-opus-4.6":
        styles[col_index["claude-opus-4.6_recall"]] = "background-color: #d9f2d9; font-weight: 700;"

    if row["best_f1_model"] == "gpt-5.4":
        styles[col_index["gpt-5.4_f1"]] = "background-color: #d9f2d9; font-weight: 700;"
    elif row["best_f1_model"] == "claude-opus-4.6":
        styles[col_index["claude-opus-4.6_f1"]] = "background-color: #d9f2d9; font-weight: 700;"

    return styles


comparison_styler = comparison_df.style.format(
    {
        "gpt-5.4_precision": "{:.2g}",
        "claude-opus-4.6_precision": "{:.2g}",
        "gpt-5.4_recall": "{:.2g}",
        "claude-opus-4.6_recall": "{:.2g}",
        "gpt-5.4_f1": "{:.2g}",
        "claude-opus-4.6_f1": "{:.2g}",
    }
).apply(highlight_best, axis=1)

display(comparison_styler)


,target,gpt-5.4_precision,claude-opus-4.6_precision,best_precision_model,gpt-5.4_recall,claude-opus-4.6_recall,best_recall_model,gpt-5.4_f1,claude-opus-4.6_f1,best_f1_model
0,src,0.83,0.9,claude-opus-4.6,0.99,0.93,gpt-5.4,0.9,0.92,claude-opus-4.6
1,bin,0.91,0.94,claude-opus-4.6,1,0.99,gpt-5.4,0.95,0.96,claude-opus-4.6
2,breaking,0.91,0.96,claude-opus-4.6,0.99,0.94,gpt-5.4,0.95,0.95,gpt-5.4


In [12]:
# Ground-truth category counts across the 310 unique benchmark cases.
truth_counts_df = pd.DataFrame(
    {
        "category": [
            "source breaking only",
            "binary breaking only",
            "both source and binary breaking",
            "backwards compatible",
        ],
        "cases": [
            int((groundtruth_df["truth_src"] & ~groundtruth_df["truth_bin"]).sum()),
            int((~groundtruth_df["truth_src"] & groundtruth_df["truth_bin"]).sum()),
            int((groundtruth_df["truth_src"] & groundtruth_df["truth_bin"]).sum()),
            int((~groundtruth_df["truth_src"] & ~groundtruth_df["truth_bin"]).sum()),
        ],
    }
)

total_cases = int(groundtruth_df["case"].nunique())
computed_total = int(truth_counts_df["cases"].sum())

print(f"Total unique benchmark cases in ground truth: {total_cases}")
print(f"Category counts total: {computed_total}")
display(truth_counts_df)


Total unique benchmark cases in ground truth: 310
Category counts total: 310


,category,cases
0,source breaking only,86
1,binary breaking only,20
2,both source and binary breaking,85
3,backwards compatible,119


## Variability of Responses

In [13]:
# Agreement/confidence score per case and per model.
# Confidence is the proportion of runs that match the majority response (range: 0 to 1).

def confidence_score(series: pd.Series):
    """
    Return the ratio of the majority vote response (!= None) in a series.
    :param series: A pandas Series of repeated responses.
    :returns: The majority proportion as a float in the range [0, 1]. Returns
        ``pd.NA`` if there are only null values (== None).
    """
    s = series.dropna()
    if s.empty:
        return pd.NA
    counts = s.value_counts()
    return float(counts.iloc[0] / len(s))


# Compute confidence for each model/target by case.
conf_df = all_runs.groupby("case", as_index=False).agg(
    gpt_54_src_conf=("gpt-5.4_src", confidence_score),
    gpt_54_bin_conf=("gpt-5.4_bin", confidence_score),
    claude_opus_46_src_conf=("claude-opus-4.6_src", confidence_score),
    claude_opus_46_bin_conf=("claude-opus-4.6_bin", confidence_score),
)

# Compute per-run breaking flags from raw per-run src/bin responses, then confidence by case.
run_level = all_runs[["case", "run", "gpt-5.4_src", "gpt-5.4_bin", "claude-opus-4.6_src", "claude-opus-4.6_bin"]].copy()

for c in ["gpt-5.4_src", "gpt-5.4_bin", "claude-opus-4.6_src", "claude-opus-4.6_bin"]:
    run_level[c] = run_level[c].map(to_bool)

run_level["gpt_breaking"] = run_level["gpt-5.4_src"] | run_level["gpt-5.4_bin"]
run_level["claude_breaking"] = run_level["claude-opus-4.6_src"] | run_level["claude-opus-4.6_bin"]

breaking_conf = run_level.groupby("case", as_index=False).agg(
    gpt_54_breaking_conf=("gpt_breaking", confidence_score),
    claude_opus_46_breaking_conf=("claude_breaking", confidence_score),
)

case_confidence_df = conf_df.merge(breaking_conf, on="case", how="inner").sort_values("case").reset_index(drop=True)
case_confidence_df


,case,gpt_54_src_conf,gpt_54_bin_conf,claude_opus_46_src_conf,claude_opus_46_bin_conf,gpt_54_breaking_conf,claude_opus_46_breaking_conf
0,accessModifierClazzAccessDecrease,1.0,1.0,1.0,1.0,1.0,1.0
1,accessModifierClazzAccessIncrease,1.0,1.0,1.0,1.0,1.0,1.0
2,accessModifierClazzConstructorAccessDecreaseNo...,1.0,1.0,1.0,1.0,1.0,1.0
3,accessModifierClazzConstructorAccessDecreasePr...,1.0,1.0,1.0,1.0,1.0,1.0
4,accessModifierClazzConstructorAccessDecreasePr...,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...
305,otherClazzDelete,1.0,1.0,1.0,1.0,1.0,1.0
306,otherClazzToIfaze,1.0,1.0,1.0,1.0,1.0,1.0
307,otherIfazeAdd,1.0,1.0,1.0,1.0,1.0,1.0
308,otherIfazeDelete,1.0,1.0,1.0,1.0,1.0,1.0


In [14]:
# Model-level aggregated percent agreement across all cases (src/bin only, no breaking).
# avg: mean percent agreement (macro = micro since all cases have equal runs).
# std: standard deviation of per-case agreements (spread around the mean).
# perfect_rate: proportion of cases where all 5 runs agreed unanimously.
summary_confidence_df = pd.DataFrame(
    {
        "model": ["gpt-5.4", "claude-opus-4.6"],
        "avg_src_conf": [
            case_confidence_df["gpt_54_src_conf"].mean(),
            case_confidence_df["claude_opus_46_src_conf"].mean(),
        ],
        "std_src_conf": [
            case_confidence_df["gpt_54_src_conf"].std(),
            case_confidence_df["claude_opus_46_src_conf"].std(),
        ],
        "avg_bin_conf": [
            case_confidence_df["gpt_54_bin_conf"].mean(),
            case_confidence_df["claude_opus_46_bin_conf"].mean(),
        ],
        "std_bin_conf": [
            case_confidence_df["gpt_54_bin_conf"].std(),
            case_confidence_df["claude_opus_46_bin_conf"].std(),
        ],
        "perfect_src_rate": [
            np.isclose(case_confidence_df["gpt_54_src_conf"], 1.0, rtol=0.0, atol=1e-12).mean(),
            np.isclose(case_confidence_df["claude_opus_46_src_conf"], 1.0, rtol=0.0, atol=1e-12).mean(),
        ],
        "perfect_bin_rate": [
            np.isclose(case_confidence_df["gpt_54_bin_conf"], 1.0, rtol=0.0, atol=1e-12).mean(),
            np.isclose(case_confidence_df["claude_opus_46_bin_conf"], 1.0, rtol=0.0, atol=1e-12).mean(),
        ],
    }
)
summary_confidence_df["avg_overall_conf"] = summary_confidence_df[["avg_src_conf", "avg_bin_conf"]].mean(axis=1)
summary_confidence_df["std_overall_conf"] = summary_confidence_df[["std_src_conf", "std_bin_conf"]].mean(axis=1)
summary_confidence_df["perfect_overall_rate"] = summary_confidence_df[["perfect_src_rate", "perfect_bin_rate"]].mean(axis=1)

summary_confidence_df.style.format({
    "avg_src_conf": "{:.3f}",
    "std_src_conf": "{:.3f}",
    "avg_bin_conf": "{:.3f}",
    "std_bin_conf": "{:.3f}",
    "avg_overall_conf": "{:.3f}",
    "std_overall_conf": "{:.3f}",
    "perfect_src_rate": "{:.1%}",
    "perfect_bin_rate": "{:.1%}",
    "perfect_overall_rate": "{:.1%}",
})

,model,avg_src_conf,std_src_conf,avg_bin_conf,std_bin_conf,perfect_src_rate,perfect_bin_rate,avg_overall_conf,std_overall_conf,perfect_overall_rate
0,gpt-5.4,0.980,0.079,0.997,0.030,93.2%,98.7%,0.988,0.054,96.0%
1,claude-opus-4.6,0.989,0.058,0.997,0.032,96.1%,99.4%,0.993,0.045,97.7%


In [15]:
# Reusable helper: show variability table and exact percent-agreement counts for one model.
# Variability is computed only from source and binary confidence (no breaking metric).
def show_model_variability(case_conf_df: pd.DataFrame, model_name: str, col_prefix: str):
    src_col = f"{col_prefix}_src_conf"
    bin_col = f"{col_prefix}_bin_conf"
    conf_cols = [src_col, bin_col]

    model_var_df = case_conf_df[(case_conf_df[conf_cols] < 1.0).any(axis=1)].copy()
    model_var_df["min_conf"] = model_var_df[conf_cols].min(axis=1)

    # Use tolerant equality for float values derived from ratios (e.g., 3/5 -> 0.6, 4/5 -> 0.8).
    is_eq_06 = np.isclose(model_var_df["min_conf"], 0.6, rtol=0.0, atol=1e-12)
    is_eq_08 = np.isclose(model_var_df["min_conf"], 0.8, rtol=0.0, atol=1e-12)
    model_var_df["agreement_value"] = np.select(
        [is_eq_06, is_eq_08],
        ["== 0.6", "== 0.8"],
        default="other (< 1.0)",
    )

    # Discern whether the inconsistency comes from the source check, the binary check, or both.
    is_src_variable = model_var_df[src_col] < 1.0
    is_bin_variable = model_var_df[bin_col] < 1.0
    model_var_df["variability_level"] = np.select(
        [is_src_variable & is_bin_variable, is_src_variable, is_bin_variable],
        ["src+bin", "src", "bin"],
        default="none",
    )

    model_var_df = model_var_df.sort_values(["min_conf", "case"]).reset_index(drop=True)

    summary_df = pd.DataFrame(
        [
            {
                "metric": f"{model_name}_src",
                "< 1.0": int((case_conf_df[src_col] < 1.0).sum()),
                "== 0.8": int(np.isclose(case_conf_df[src_col], 0.8, atol=1e-12).sum()),
                "== 0.6": int(np.isclose(case_conf_df[src_col], 0.6, atol=1e-12).sum()),
            },
            {
                "metric": f"{model_name}_bin",
                "< 1.0": int((case_conf_df[bin_col] < 1.0).sum()),
                "== 0.8": int(np.isclose(case_conf_df[bin_col], 0.8, atol=1e-12).sum()),
                "== 0.6": int(np.isclose(case_conf_df[bin_col], 0.6, atol=1e-12).sum()),
            },
            {
                "metric": f"{model_name}_min(src,bin)",
                "< 1.0": int((model_var_df["min_conf"] < 1.0).sum()),
                "== 0.8": int(np.isclose(model_var_df["min_conf"], 0.8, atol=1e-12).sum()),
                "== 0.6": int(np.isclose(model_var_df["min_conf"], 0.6, atol=1e-12).sum()),
            },
        ]
    )

    print(f"{model_name} variable cases (src/bin only): {len(model_var_df)}")
    display(model_var_df[["case", src_col, bin_col, "min_conf", "agreement_value", "variability_level"]])

    print(f"\n{model_name} variability counts by metric and exact agreement values:")
    display(summary_df)

    return model_var_df, summary_df


# Claude-only variability output in this cell.
claude_variability_df, claude_variability_summary = show_model_variability(
    case_confidence_df,
    model_name="claude-opus-4.6",
    col_prefix="claude_opus_46",
)

claude-opus-4.6 variable cases (src/bin only): 14


,case,claude_opus_46_src_conf,claude_opus_46_bin_conf,min_conf,agreement_value,variability_level
0,dataTypeClazzMethodParamUnboxing,0.6,1.0,0.6,== 0.6,src
1,dataTypeClazzMethodReturnTypeNarrowing,0.6,1.0,0.6,== 0.6,src
2,dataTypeIfazeConstantNarrowing,1.0,0.6,0.6,== 0.6,bin
3,dataTypeIfazeConstantWidening,1.0,0.6,0.6,== 0.6,bin
4,genericsClazzConstructorTypeDeleteN,0.6,1.0,0.6,== 0.6,src
5,genericsIfazeMethodTypeAddN,0.6,1.0,0.6,== 0.6,src
6,genericsWildcardsClazzMethodParamAdd,0.6,1.0,0.6,== 0.6,src
7,dataTypeClazzFieldBoxing,0.8,1.0,0.8,== 0.8,src
8,dataTypeIfazeMethodReturnTypeBoxing,0.8,1.0,0.8,== 0.8,src
9,exceptionIfazeMethodThrowCheckedAdd,0.8,1.0,0.8,== 0.8,src



claude-opus-4.6 variability counts by metric and exact agreement values:


,metric,< 1.0,== 0.8,== 0.6
0,claude-opus-4.6_src,12,7,5
1,claude-opus-4.6_bin,2,0,2
2,"claude-opus-4.6_min(src,bin)",14,7,7


In [16]:
# GPT-5.4-only variability output (src/bin only; helper defined in previous cell).
gpt_variability_df, gpt_variability_summary = show_model_variability(
    case_confidence_df,
    model_name="gpt-5.4",
    col_prefix="gpt_54",
)

gpt-5.4 variable cases (src/bin only): 25


,case,gpt_54_src_conf,gpt_54_bin_conf,min_conf,agreement_value,variability_level
0,accessModifierClazzMethodAccessIncreaseNonToPr...,0.6,1.0,0.6,== 0.6,src
1,accessModifierClazzMethodAccessIncreaseNonToPu...,0.6,1.0,0.6,== 0.6,src
2,accessModifierClazzMethodAccessIncreasePrivate...,0.6,1.0,0.6,== 0.6,src
3,accessModifierClazzMethodAccessIncreaseProtect...,0.6,1.0,0.6,== 0.6,src
4,genericsClazzMethodTypeBoundsDeleteN,0.6,1.0,0.6,== 0.6,src
5,genericsClazzMethodTypeBoundsGeneralization,0.6,1.0,0.6,== 0.6,src
6,genericsIfazeMethodTypeBoundsDeleteN,0.6,1.0,0.6,== 0.6,src
7,genericsWildcardsClazzMethodParamLowerBoundsDe...,0.6,1.0,0.6,== 0.6,src
8,membersClazzFieldConstantAdd,0.6,1.0,0.6,== 0.6,src
9,membersClazzMethodAdd,0.6,1.0,0.6,== 0.6,src



gpt-5.4 variability counts by metric and exact agreement values:


,metric,< 1.0,== 0.8,== 0.6
0,gpt-5.4_src,21,11,10
1,gpt-5.4_bin,4,3,1
2,"gpt-5.4_min(src,bin)",25,14,11


In [17]:
# Intersection of inconsistent cases between Claude and GPT (src/bin variability).
claude_inconsistent_cases = set(claude_variability_df["case"])
gpt_inconsistent_cases = set(gpt_variability_df["case"])

intersection_cases = sorted(claude_inconsistent_cases & gpt_inconsistent_cases)

# Bring in each model's variability_level (src / bin / src+bin) for the shared cases,
# so we can see whether the inconsistency is at the source or binary level per model.
intersection_df = pd.DataFrame({"case": intersection_cases})
intersection_df = intersection_df.merge(
    claude_variability_df[["case", "variability_level"]].rename(
        columns={"variability_level": "claude_level"}
    ),
    on="case",
    how="left",
).merge(
    gpt_variability_df[["case", "variability_level"]].rename(
        columns={"variability_level": "gpt_level"}
    ),
    on="case",
    how="left",
)

print(f"Claude inconsistent cases: {len(claude_inconsistent_cases)}")
print(f"GPT inconsistent cases: {len(gpt_inconsistent_cases)}")
print(f"Intersection (both inconsistent): {len(intersection_cases)}")

intersection_df


Claude inconsistent cases: 14
GPT inconsistent cases: 25
Intersection (both inconsistent): 3


,case,claude_level,gpt_level
0,dataTypeIfazeConstantWidening,bin,bin
1,genericsIfazeMethodTypeBoundsDeleteN,src,src
2,genericsIfazeMethodTypeBoundsGeneralization,src,src


In [18]:
# Conclusion: for the cases where both models are inconsistent, is the inconsistency
# happening at the same level (source vs. binary) for both models, or different levels?
intersection_df["same_level"] = intersection_df["claude_level"] == intersection_df["gpt_level"]

level_agreement_counts = intersection_df["same_level"].value_counts().rename(
    {True: "same level for both models", False: "different level per model"}
)

print("Breakdown of shared inconsistent cases by source/binary level:")
display(intersection_df.groupby(["claude_level", "gpt_level"]).size().rename("cases").reset_index())

print("\nSame level vs. different level across models:")
display(level_agreement_counts)

n_src = int((intersection_df["claude_level"] == "src").sum())
n_bin = int((intersection_df["claude_level"] == "bin").sum())
n_both = int((intersection_df["claude_level"] == "src+bin").sum())

Breakdown of shared inconsistent cases by source/binary level:


,claude_level,gpt_level,cases
0,bin,bin,1
1,src,src,2



Same level vs. different level across models:


same_level
same level for both models    3
Name: count, dtype: int64

## Resources Use

In [19]:
# Descriptive statistics per case per model for input/output tokens and request time.

models = ["gpt-5.4", "claude-opus-4.6"]
metrics = ["input_tokens", "output_tokens", "request_time"]

metric_columns = {
    metric_name: {m: f"{m}_{metric_name}" for m in models}
    for metric_name in metrics
}

# One value per case/model/metric: mean across runs.
agg_map = {
    f"{m}_{metric_name}_per_case": (metric_columns[metric_name][m], "mean")
    for m in models
    for metric_name in metrics
}

per_case_metrics_df = all_runs.groupby("case", as_index=False).agg(**agg_map)


def describe_series(s: pd.Series):
    x = s.dropna()
    return {
        "min": x.min(),
        "q1": x.quantile(0.25),
        "median": x.median(),
        "mean": x.mean(),
        "q3": x.quantile(0.75),
        "max": x.max(),
        "std": x.std(),
    }


rows = []
for m in models:
    for metric_name in metrics:
        col = f"{m}_{metric_name}_per_case"
        stats = describe_series(per_case_metrics_df[col])
        rows.append({"model": m, "metric": metric_name, **stats})

descriptive_stats_df = pd.DataFrame(rows)

print("\nDescriptive statistics per case per model (case-level mean across runs):")
display(descriptive_stats_df.style.format({
    "mean": "{:.2f}",
    "median": "{:.2f}",
    "min": "{:.2f}",
    "max": "{:.2f}",
    "q1": "{:.2f}",
    "q3": "{:.2f}",
    "std": "{:.2f}",
}))


Descriptive statistics per case per model (case-level mean across runs):


,model,metric,min,q1,median,mean,q3,max,std
0,gpt-5.4,input_tokens,570.00,628.00,639.50,645.77,652.00,806.00,32.97
1,gpt-5.4,output_tokens,59.80,223.60,361.40,584.09,660.90,3836.80,604.65
2,gpt-5.4,request_time,1870.60,4312.17,6158.93,8929.80,10171.26,47886.04,7690.59
3,claude-opus-4.6,input_tokens,710.00,795.00,815.00,824.36,837.00,1010.00,48.71
4,claude-opus-4.6,output_tokens,30.00,46.10,128.30,180.47,252.25,1563.60,181.35
5,claude-opus-4.6,request_time,2029.40,2823.48,5078.97,6961.35,10007.38,35728.56,5022.18


In [20]:
# Descriptive statistics across all runs (no case aggregation).
rows_all_runs = []
for m in models:
    for metric_name in metrics:
        col = metric_columns[metric_name][m]
        stats = describe_series(all_runs[col])
        rows_all_runs.append({"model": m, "metric": metric_name, **stats})

descriptive_stats_all_runs_df = pd.DataFrame(rows_all_runs)

print("Descriptive statistics across all runs (no case aggregation):")
display(descriptive_stats_all_runs_df.style.format({
    "min": "{:.2f}",
    "q1": "{:.2f}",
    "median": "{:.2f}",
    "mean": "{:.2f}",
    "q3": "{:.2f}",
    "max": "{:.2f}",
    "std": "{:.2f}",
}))

Descriptive statistics across all runs (no case aggregation):


,model,metric,min,q1,median,mean,q3,max,std
0,gpt-5.4,input_tokens,570.00,628.00,640.00,645.77,652.00,806.00,33.29
1,gpt-5.4,output_tokens,44.00,201.25,329.50,584.09,531.00,4841.00,717.83
2,gpt-5.4,request_time,1375.92,3971.51,5745.95,8929.80,9320.42,133726.16,9542.34
3,claude-opus-4.6,input_tokens,710.00,795.00,815.00,824.36,837.00,1010.00,49.67
4,claude-opus-4.6,output_tokens,27.00,46.00,121.50,180.47,250.75,2156.00,198.33
5,claude-opus-4.6,request_time,1803.60,2675.43,4590.03,6961.35,9980.72,47013.48,5632.48
